# Глава 11. LLM-агенты и агентские системы

## Введение
До этого момента всё наше общение с LLM укладывалось в простую схему: мы подаем на вход текст и в ответ получаем текст. Модель выступает в роли справочной. Пока речь идёт о том, чтобы объяснить, перевести или переписать, большего и не требуется. Но чтобы можно было использовать. Уровня абстрактного 

Сначала модели размышлять, Затем использованию.

ИИ Агент — это генеративная языковая модель, облатающая субъектностью, то есть запущенная в условно бесконечном цикле. Модель рассуждает, использует внешние инструменты, оценивает и планирует дальнейшие шаги, пока не достигнет поставленной ей цели

<img src="img/agent.jpg" width=500>

## Рассуждающие модели
Ранние LLM работали по принципу "вопрос - ответ" и если на простых задачах это хорошо работает, то на задачах, требующих многошагового логического рассуждения, модель часто ошибается, пытаясь сразу «перепрыгнуть» к ответу. Оказалось, что дело не столько в нехватке знаний, сколько в отсутствии у модели «черновика»: не было пространства, чтобы развернуть ход мыслию. 

__Reasoning__ - это способность модели не выдавать ответ сразу, а разворачивать промежуточные шаги рассуждения перед финальным выводом.

В 2022 году авторы из Google [[Wei et al., 2022]](https://arxiv.org/abs/2201.11903) обнаружили, что если попросить модель не выдавать ответ сразу, а сначала проговаривать промежуточные шаги, качество ее ответов на сложных логических задачах резко растёт. Они это делали через Few-shot промптинг: модели явно показывали пример, какого формата они ожидают ответ. Этот вид промптинга назвали **Chain-of-Thought (CoT)** размышление

Промежуточные шаги предоставляют пространство, в котором сложная задача декомпозируется на простые подзадачи. Кроме того, каждый дополнительный токен рассуждения — это, дополнительная порция вычислений, потраченная на задачу

Лучше всего для тестирования рассуждения подходят математические задачи - есть строгое поэтапное ветвление логики и один объективный конечный ответ

Чуть позже ([Kojima et al., 2022](https://arxiv.org/abs/2205.11916)) показали, что даже не нужно добавлять в промпт few shot примеры, достаточно одной фразы вроде «давай рассуждать шаг за шагом», чтобы запустить тот же самый эффект. Это поразительно, но одна эта фраза увеличивает точность модели в 2-4 раза на логических задачах. А с примерами прирост еще выше

Важно отметить, что CoT помогает только там, где есть что декомпозировать (арифметика, логика, многоступенчатые выводы), но может вредить на простых задачах, где лишние рассуждения лишь добавляют шум и возможности ошибиться.

Одна цепочка рассуждений ненадёжна: модель может уверенно пойти по ошибочному пути. ([Wang et al., 2022](https://arxiv.org/abs/2203.11171)) предлагает генерировать не одну цепочку рассуждений, а сразу много, за счёт случайности при сэмплировании. затем выбрать ответ голосованием большинства. Ключевая интуиция: правильное рассуждение обычно сходится к одному и тому же ответу разными путями, тогда как ошибки «расходятся» и не складываются в большинство. Такой подход назвали __Self-Consistency__ размышление

Работа ([Yao et al., 2023](https://arxiv.org/abs/2305.10601)) переосмысляет рассуждение как поиск по дереву. Вместо линейной цепочки модель на каждом шаге порождает несколько вариантов продолжения рассуждения, оценивает их перспективность, углубляется в удачные ветви и откатывается от тупиковых. Модель получила название __Tree-of-thoughs__.

<img src="img/cot4.png" width=500>

пошагово, в промпте явно прописывают выполняй ровно один шаг, так модель понимает, когда ей нужно остановиться для переоценки траектории. 

В роли оценщика (Verifier или Model-as-a-judge) обычно выступает та же самая модель, она отвечает на вопрос: похоже ли, что рассуждение приведет к правильному ответу? Траекторию можно оценивать в абсолютных значениях ("sure" - да, уже пришли к ответу, "impossible" - ветка рассуждения плохая ни к чему не приведет, "maybe" - пока нельзя сказать); а можно ранжировать (отсортируй по потенциалу и мы отберем топ-5 перспективных цепочек).

Самооценка своей же моделью страдает от проблемы предвзятости (Self-Correction Fallacy). Во-первых, искуственный интеллект традиционно плохо калибрует свою уверенность (как в целом и люди). Во-вторых, у ИИ есть склонность "поддакивать" пользователю, если контексте есть утверждение X, модель считает, что логичнее ему следовать, чем опровергать

Как с этим можно бороться: 
- провести k независимых проверок каждой цепочки, агрегируя результат с помощью голосования (majority vote)
- грамотно составлять промпты на оценку: не "как ты оцениваешь рассуждение?", а просили проделать аналитическубю работу, провести тесты и сравнения<br>
- разделять "модель генератор" и "модель критика"
- в соверменных моделях используется Reinforcement Learning

Селекция траеткорий для продолжения происходит либо через перебор всех нетупиковых ветвей (если ресурсы позволяют), либо с помощью Beam Search: в каждый момент времени работаем с K наиболее перспективных вариантов продолжений

Если дать модели возможность смыкать разные траектории, то вместо дерева получится ациклический граф. Такой вариант назвали __Graph-of-thoughts__. Идея в том, что перспективные траектории могут дать синергию при обэединении.

Часто модели ориентированы на действие. Вместо плана пишется код. Так сделали авторы __Program-of-Thoughts__.

Для генерации few-shot примеров, когда нет желания или возможности собирать примеры вручную. Auto-CoT

Во всех последних методах процесс размышления гибкое. Все эти подходы можно описать зонтичным теримном Test-Time Scaling.

качество растёт, если позволить модели «думать» дольше — порождать длинные внутренние рассуждения перед ответом

Как этому обучают — отдельный сюжет. Ключевой приём — RL с проверяемыми наградами (RLVR, RL with verifiable rewards): модель тренируют на задачах, где правильность ответа можно автоматически проверить (математика, код), и награждают за верный итоговый результат, не предписывая, *как именно* рассуждать. Модель сама «открывает» эффективные стратегии рассуждения 

### Рассуждения в DeepSeek-R1
В сентябре 2024 OpenAI выпустила свою модель o1, которая стала первой крупной моделью для выполнения сложных многошаговых рассуждений и обученной с помощью подкрепления (RL). На тестах модель достигла уровня лучших выпускников школ, что было прорывом. Но проблема в том, что она оставалась закрытой и проприетарной

В начале 2025 года китайский стартап DeepSeek выпустил новую версию своей одноименной модели, которая также была заточена под рассуждение [DeepSeek-R1, 2025](https://arxiv.org/abs/2501.12948). Но в отиличие от OpenAI, они сделали модель открытой, а также опубликовали техническое описание процесса обучния.

Появление модели DeepSeek-R1 стало важной вехой, так как она не только догнала (а в некоторых аспектах превзошла) o1 по тестам, но и сделала технологии доступными для всего исследовательского сообщества, что дало толчок развитию области рассуждений. 

Успех модели впечатлил инвесторов, особенно заявленная стоимость дообучения (300 тысяч долларов), которая была в разы ниже OpenAI. В моменте новая модель обрушила акции NVidia главного производителя дорогостоящих чипов на 20%. Позднее правда стоимость откатились назад.

Но самое главное, модель показала, что выдающиеся способности к рассуждению могут возникнуть спонтанно в процессе обучения с подкреплением, без необходимости какого-либо дорогостоящего обучения на примерах

Одним из поразительных эффектов был так называемый "A-ha moment", когда в процессе размышления модель в какой-то момоент буквально говорит "подождите-ка... я поняла" , понимает, как правильно решить задачу, зачеркивает все написанное ранее и пишет правильное решение.

СМИ назвали это проявлением осознанности и подавали как способность искусственного интеллекта к саморефлексии, хотя научное сообщество сошлось, что антропоморфная трактовка здесь не совсем корректна: это хоть и впечатляющая, но просто иллюзия когнитивного процесса.

Это пример эмерджентности - свойства появлению у модели новых способностей, которые не закладывались изначально и проявляются по мере роста сложности системы

---

## Свойства агентности

### Реактивность
Рассуждение, замкнутое внутри модели - это монолог. Такая модель все тот же black box, просто тратящая больше токенов на этапе генерации. Нам же хочется, чтобы рассуждение было менее линейным, и более интеракивным - могло влиять на внешнюю среду и реагировало на изменения. Свойство подстраиваться под изменение среды будем называть реактивностью

([Yao et al., 2022](https://arxiv.org/abs/2210.03629)) из Google описали паттерн рассуждения **ReAct**, который реализовывал ровно такой подход: чередовал рассуждение и действие. Название ReAct - склеено из *Reasoning* и *Acting*. Алгоритм рассуждения состоит из слдедующих шагов:
1) модель рассуждает («чтобы ответить, мне нужно узнать X»)
2) выбирает действие (например, поиск)
3) получает наблюдение (результат поиска)
4) возвращается к 1: повторяет рассуждение, но уже с учётом нового факта
5) генерирует финальный ответ

Важно понимать, что реактивность важна не всех задачах, а только там, где внутренних знаний модели не достаточно и нужно обогащение к внешней информацией

В качестве модели использовалась обычная языковая модель, а обучение делали в zero-shot режиме: через показ примеров. Чтобы моднль могла обновлять контекст данными, ей дали доступ к трем функциям подгрузки информации из Wiki: "[seach entity]" (найти информацию по термину) "[lookup string]" (найти строку в документе), "[finish answer]" (оформить ответ). По сути первая модель генерации, которая во-первых дала толчок использованию инструментов, а во-вторых Retrieval Generation систем на базе LLM. 

---

### Использование инструментов
У людей первые механические орудия (каменное рубило) появились около 3 млн лет назад. Они фактически заменили человеку зубы, открыв доступ к высококалорийной пище. предположительно главной предпосылкой к их появлению стало развитие прямоходждения (Homo Erectus вышли из леса), развитие кистей, и появление абстрактоного мышления (орудие нужно еще изготовить). А высокая социализированность по сравнению с другими видами позволила поствить это умение "на конвейер". В результате вектор эволюции сместился: вместо статического развития у вида Homo появилась возможность изменять окружающую среду

В контексте языковых моделей инструмент (Tool) - это любая внешняя по отношению к модели возможность изменить контекст. Самые первые инструментов: поиск (Web или база данных), калькулятор, прогноз погоды и прочее. Как правило инструменты параметризованные - запрос

Паттерн ReAct описал, как можно рассуждение действиями, но не хватало универсального механизма, который описал бы, как подключать к модели произвольные инструменты. ([Schick et al., 2023](https://arxiv.org/abs/2302.04761)) решили восполгнить это упущение и масштабировали подход ReAct, перейдя от zero-shot к обучению. Они назвали свою модель **Toolformer**, которая на какое-то время стала стандартом применения инструментов.

Обучение реализуется это в два шага. На первом модели показывается несколько few-shot примеров, как можно использовать инструмент и та пытается вставлять везде в тексте, где это выглядит разумным. Далее она смотрит на внутренние метрики, (как правило перплексию), и если вставка ощутимо растит метрику справа от нее, значит здесь инструмент полезен - помечается, как положительный пример

На втором шаге языковая модель полноценно обучается по сгенерированному на предыдущем шаге синтетическому датасету. В целях экономии авторы не стали делать бесконечный цикл, как в ReAct, ограничились разовым обогащением

<img src="img/toolformer.png" width=750>

RL для рассуждений:
- __STaR__<br>Self-Taught Reasoner (Zelikman et al., 2022) — берём задачу с однозначным ответом. Генерируем одну траекторию жадным декодированием. Если ответ верный — в датасет. Если нет — переспрашиваем, подав правильный ответ в промпт, получаем траекторию задним числом, кладём в датасет уже без подсказки. Собираем SFT-датасет, дообучаем базовую модель (не предыдущую итерацию). Повторяем.
- __ReST__<br>(Gulcehre et al., 2023; Singh et al., 2023) — Берем задачу с однозначным ответом. Генерируем N траекторий (разнообразие обеспечивается температурой генерации). Оставляем приведщие к праильномк ответу, из них собираем SFT датасет и дообучаем на нем модель
- __RFT__<br>Метод Rejection Sampling Fine-Tuning, (Yuan et al., 2023) — то же, но сэмплируем много вариантов и фильтруем по проверяемому критерию
- __RAFT__<br>(Reward-rAnked FineTuning, Dong et al., 2023)
- Исторический предок в RL-литературе: expert iteration (Anthony et al., 2017, та же идея испольщуется в AlphaZero)

**HuggingGPT** от Microsoft ([Shen et al., 2023](https://arxiv.org/abs/2303.17580)) показал, что можно использовать LLM как дирижера __LLM-as-a-Controller__: языковая модель выступает диспетчером, который разбивает задачу и распределяет подзадачи между множеством специализированных моделей и инструментов. Пример - когда в задании много модальностей

---

### Планирование
Для коротких задач простого последовательного рассуждения часто бывает достаточно. Но иногда задача слишком масштабна и нужен специальный механизм ее координации. В роли такого механихма выступает планирование.

Планимрование необходимо когда:
а) ландшафт решения слишком сложен - задача поставлена слишком абстрактно или наоборот задано слишком много конкурирующих требований
б) высок риск неправильных действий, сменить траекторию решения непросто

В этом случае лучше продумать ход решения заранее, это будет дешевле

Пример такой задачи - оформление командировки сотрудника из Москвы в Мехико на конкретные даты в соотвествии с выделенным бюджетом. У сотрудника есть две банковские карты: одна заблокирована для международных операций, вторая — с лимитом в 50 000 рублей. Агент при этом умеет через API проверять цены, искать визовые требования к каждой стране, может проверять остаток на карте.

Решение понятное - нужно каким-то образом внедрить этап планирования как отдельный компонент, и отделить его от выполнения.

Соотнести в рамках модели планирование и исполнение можно двумя способами:
- сначала составить план, потом выполнять
- циклически корректировать план на ходу выполнения

Первый подход проще и экономичнее, так как достаточно всего одной итерации планирования, но вместе с тем выше риск ошибки, если план окажется плохим. В свою очередь второй подход гибче, но выше риск погрязнуть в корректировках. Тут уместна аналогия с waterfall и agile методологиями разработки, логика примерно та же.

Самый простой пример такого планирования - паттерн Plan-and-Solve. [(Wang et al., 2023)](https://arxiv.org/abs/2305.04091)) по аналогии с Chain-of-thought просто попросили модель в промпте "сначала построй план, а потом его реализуй". Уже этого было достаточно, чтобы получить более хорошие решения на задачах, треующих планирования.

Кроме того, при составлении плана модель часто декомпозирует задачу на список независимых подзадач и имея этот список, их можно запускать на выполнение параллельно, что делает генерацию более эффективной. К этому же классу можно отнести модели *Least-to-most*, которая ранжирует подзадачи по сложности, *ReWOO*, и рассмотренный выше *HuggingGPT*.

В большинстве случаев агенты ориентированы на выполнение каких-либо действий, поэтому исполнитель - это не обязательно языковая модель, может выступать и детерминированный инструмент, как например в модели LLM+P [(Liu et al., 2023)](https://arxiv.org/abs/2304.11477),  Они составляют план в стандартизированном формате [PDDL](https://en.wikipedia.org/wiki/Planning_Domain_Definition_Language), и модель его выполняет. ProgPrompt

Второй подход напоминает расмотренный выше цикл ReAct - планируется и выполняется один шаг, переоценивается и цикл повторяется. К нему можно отнести Successive prompting, Self-Ask, где модель задает уточняющие вопросы

К этому же классу отнести модели поиска по дереву планов, как в рассуждающих моделях. Мы рассамтривали выше Tree of Thought. Graph-of-thought заменяет деерво траекторий на граф траекторий. RAP (Reasoning via Planning) , MCTS (Monte Carlo Tree Simluation)

И также есть модели, в которых пытаются объединить оба подхода: выполнение задачи идет по четкому плану, но сам план может периодически пересматриваться. Сюда например, можно отнести модели LLM+P.

Похоже на итеративность ReAct, но ReAct - это про итеративность для обогащения внешними данными, а здесь про итеративность для перепланирования логики получения ответа

В общем случае планирование - опциональный элемент языкового агента. Ближе всего он к Research Mode в чат-ботах, а также в системах типа Claude Code или Codex, где задача обычно требует выполнения множества шагов и следования выбранной траектории

---

### Память (Memory)
Языковые модели не хранят состояния между вызовами, взаимодейтсвие реализуется через контекст, который каждый раз читается полностью заново (а KV кэш?). Но контекстное окно конечно. При всех оптимизациях не бывает больше 1M токенов. Это фундаментальное ограничение: без памяти агент не может вести длинную задачу или помнить о прошлых взаимодействиях

В контексте языковых моделей выделяют следующие виды памяти:

- Краткосрочная память (Short-term Memory, STM): Информация в текущем контекстном окне, например, история последних сообщений в чате. Она быстрая, но ограничена максимальным размером контекста, обычно от 4k до 200k токенов

- Долгосрочная память (Long-term Memory, LTM): Информация, которая сохраняется за пределами контекстного окна и может быть использована в будущих сеансах. Хранится в базах данных, векторных хранилищах или файлах.

- Эпизодическая память (Episodic Memory): Хранит воспоминания о конкретных событиях и опыте (например, "Во вторник пользователь попросил перенести демонстрацию").

- Семантическая память (Semantic Memory): Хранит факты и общие знания, независимые от времени и контекста (например, "Пользователь предпочитает встречи во второй половине дня").

- Процедурная память (Procedural Memory): Хранит информацию о том, как выполнять задачи — инструкции, рабочие процессы и правила использования инструментов.

- Рабочая память (Working Memory): Подмножество информации, которая в данный момент загружена в контекстное окно и доступна для немедленного использования

[(Liu et al., 2023)](arXiv:2307.03172) изучили, как именно используется контекстная память, оказалось, что эффективность доступа крайне неравномерна. Этот эффект получил название «Lost in the Middle».

В рекуррентных сетях память - это агрегат контекста. 

Базовые приёмы 
- суммаризация истории (сжать прошлый диалог, чтобы он влез в окно),
- векторные хранилища (сохранять факты и доставать релевантные по семантическому поиску — основа RAG)
- аккуратное управление контекстным окном

([Packer et al., 2023](https://arxiv.org/abs/2310.08560)) вдохновились тем, как организована память в ОС и предложили **MemGPT** одну из первых моделей, реализующих механизм внешней памяти. Здесь LLM явно реализует двухуровневую модель памяти (оперативная vs постоянная) и сама решает, что держать в «оперативном» контексте, а что вытеснить во внешнее хранилище и подгрузить обратно при необходимости. Можно провести аналогию с [виртуальной памятью](https://en.wikipedia.org/wiki/Virtual_memory) и страничной подкачкой (page swap)

<img src="img/memgpt1.png" width=500>

Наличие памяти позволяет работать с задачами длиннее, чем его контекстное окно

---

### Рефлексия и самообучение
Важное свойство языковых агентов - способность критиковать свои решения и улучшать их, не дожидаясь внешнего стимула.  Если планирование, которое мы рассматривали выше, это активность перед генерацией, то рефлексия - это активность после генерации ответа. Цель рефлексии - повторно запустить генерацию с исправлением. Проводя аналогию с разработкой ПО, планировние - это выбор задач из бэклога перед началом спринта, а рефлексия это ретроспектива в конце спринта

(Madaan et al., 2023) описали вероятно самое простое решение, после генерации ответа запустить еще один цикл с промптом "покритикуй свое решение". Подход назвали __Self-Refine__. Он хорошо подходит для например, оценки стиля. Но для проверки фактичности не подходит

Проблема с саморефлексией - если модель оценивает сама себя, она предвзята. Распределения идентичны

Частично можно исправить, попросив модель оценить не смотри на оригинальный ответ. Этот подход назвали __Chain-of-Verification__. Идея в том, что сначала модель генерирует черновик ответа, затем по этому черновику формулирует набор проверочных вопросов и получает ответы на них. После этого компонует ответ

Пример : «Назови нескольких политиков, родившихся в Нью-Йорке»
Черновик: Хиллари Клинтон, Дональд Трамп, Майкл Блумберг.
Сгенерированные вопросы:
1. Где родилась Хиллари Клинтон?
2. Где родился Дональд Трамп?
3. Где родился Майкл Блумберг?
Ответы (каждый — отдельный вызов, черновика в контексте нет):
1. Чикаго, Иллинойс          ← расхождение
2. Куинс, Нью-Йорк           ← подтверждено
3. Бостон, Массачусетс       ← расхождение

От проблемы идентичных распределений можно частично, если разнести исполнителя и верификатор по разным моделям. ([Shinn et al., 2023](https://arxiv.org/abs/2303.11366)) предложили добавлять критику промптом. Модель получила название **Reflexion**. Дает +11%



тесты / компиляция
CRITIC - проверяет ответ фактовым запросом
отдельная LLM оценивает



<img src="img/reflection2.png" width=300>

**Voyager** ([Wang et al., 2023](https://arxiv.org/abs/2305.16291))<Br>довёл идею до пожизненного обучения на примере агента в Minecraft. Три механизма работают вместе: автоматический учебный план (агент сам ставит себе посильно усложняющиеся цели), библиотека навыков (удачные решения сохраняются как переиспользуемый код и комбинируются в более сложные) и обратная связь от среды-песочницы. Так агент постепенно накапливает компетенцию, а не решает каждую задачу с нуля

Агент становится самоулучшающимся в пределах сессии или «жизни», а библиотека навыков превращает разовые решения в композиционную, переиспользуемую компетенцию

---

<img src="img/agent_papers.jpg" width=500>

---



# Мультиагентные системы
Если агенты стали настолько умными, что способны автономно решать заадчи, то почему бы не запустить сразу команду агентов? Ведь на бытовом уровне, чтобы запустить амбициозный проект, нужна команда хороших специалистов. Как бы хороши они не были по отдельности, их навыков просто не хватит. Один агент может перегружается задачами, путать роли (если их много), терять контроль над длинным процессом. __Мультиагентная система__ - множетсво языковых агентов, работающих вместе в координации для выполнения глобальной цели

Аргументы в пользу использования команды агентов:
- способ прочитать больше, чем влезает в контекст
- можно распараллелить выполнение
- специализация агентов:
    - безопасность - агент выполняет строго определенный набор действий
    - экономия - для простых задач требуется меньше токенов
    - изоляция ошибок - агент можно перезапустить

Независимость однако порождает разногласие и чем больше в системе агентов, тем больше нужно уделять внимания их координации. В целом системы, ориентированные на чтение проще, чем системы ориентированные на запись (Cognition vs Antrophic). 

Можно предположить, что много агентов улучшает "широту" мышления (Divergent Thinking) - якобыкоманда агентов может предложить больше вариантов решения, чем отдельный агент. Но на практике проихсодит скорее наоборот. [(Wang et al, 2024)](https://arxiv.org/abs/2406.06461) показали, что чем больше координируют свое рассуждение, пытаясь прийти к ответу, тем сильнее падает энтропия (разнообразие) ответов. Это происходит из-за того, что . Для сравнения в методе рассуждения self-consistency такого не происходит

Другой аргемент - повышает точность ответов (improves factuality & reasoning). Это действительно так, но незщначитеолтьнро, тот же Chain-of-thought достигает такого же результата дешевле

Что лучше валидирует ответ (Validation)

Мультиагентые системы - один из главных трендов развития ИИ начиная с 2025 года. Стоит отметить, что более поздние работы показали, что преимущество команды агентов перед индивидуальными исполнителями не так очевидно. Например, здесь [(Tran et al, 2026)](https://arxiv.org/pdf/2604.02460) показывали, что при фиксированном бюджете одноагентная конфигурация может быть ни чуть не хуже. Или здесь [(Xu et al, 2026)](https://arxiv.org/pdf/2601.12307) авторы исследуют важность узкой специализации и опровергают её, показав что ту же систему можно реализовать одной моделью, назначая ей роли промптами

Однако и риски растут. Появляются каскадные ошибки, становится возможным prompt injection, работа иногда зацикливается

Поэтому важна координация. Как определяется приоритеность команд? Например, приказ смежного агента противоречит системному промпту. Как определяется порядок выполнения в мультагентных системах: может жестко по графу. пример - игра в мафию
- агент сам определяет (push режим)<br>игра в мяч
- GroupChatManager - коорднатор<br>модератор на панельной сессии
- по условию / триггеру<br>

Агенты - это как правило отдельные сессии или микросервисы, выполняют работу параллельно, но важно отметить, что их координация дискретна, нужно должаться полного выполния перед следующим шагом

---

### AutoGen
Одна из первых работ, описавщая - фреймворк **AutoGen** от Microsoft. ([Wu et al., 2023](https://arxiv.org/abs/2308.08155)) предложили как можно организовать совместную работу множества агентов, просто дав им возможность коммуницировать на естественном языке

Работа популяризировала термины Conversable Agent (диалоговый агент) - модель, готовая коммуницировать с внешним миром через язык и Conversable Programming (диалоговое программирование) - способ координации работы между агентами при решении ими распределенной задачи, когда задачи ставятся текстом

В работе вводят несколько классов агентов по 
- AssistantAgent - "мозги" для генерации рассуждения
- UserProxyAgent - "руки" для автономного выполнения действий. Моделирует "что бы сделал пользователь"
- GroupChatManager - (опционально) координатор, выбирает, кто будет следующий действовать

<img src="img/autogen1.png" width=600>

Модель развитвается. В более поздней версии реализовали модель вычислений Actor-model

> [Actor Model](https://en.wikipedia.org/wiki/Actor_model) - это математическая модель асинхронных вычислений. Актор - это объект, который умеет а) принимать сообщения б) отправлять сообщения в) порождать дургих акторов г) выполнять какое-то действие<br>
Модель разрабатывали еще с 1973 года. На основе нее были реализованы многие инженерные фреймворки для разработки Message-driven applications, например, фреймворк [Akka](https://en.wikipedia.org/wiki/Akka_(toolkit)) для Java<br>
Похоже на концепцию микросервисов: вычисление тоже бъется на отдельные независимые куски, запускаемые асинхронно, но разница в уровне детализации. Akka работает на уровне отдельных объектов

### MetaGPT

В другой работе **MetaGPT** ([Hong et al., 2023](https://arxiv.org/abs/2308.00352)) принцип тот же, но тут разделение труда более стандартизированный. Там пошли от метафоры организации: каждый агент выполняет свою роль, как это бывает в софтверной компании: продакт-менеджер, архитектор, инженер

<img src="img/metagpt1.png" width=500>

### Оркестрация
По мере усложнения систем линейных цепочек становится мало — нужен переход к управляемым графам состояний. Графовая оркестрация (типичный представитель — [LangGraph](https://github.com/langchain-ai/langgraph)) описывает агента как граф: узлы — это шаги, рёбра — переходы, есть общее состояние и допустимы циклы. Это даёт явный контроль над тем, что и в каком порядке происходит.

Хороший словарь паттернов задаёт статья Anthropic [«Building Effective Agents»](https://www.anthropic.com/engineering/building-effective-agents). Она проводит важное различие между workflow (заранее заданные маршруты, по которым модель ведут жёстко) и собственно агентами (модель сама управляет своим процессом). И описывает базовые композиционные паттерны: chaining (последовательная цепочка), routing (маршрутизация запроса в нужную ветку), параллелизация, оркестратор-исполнители, оценщик-оптимизатор

### Стандарты подключения (MCP)
С ростом популярности использования инструментов, встал вопрос интеграции: каждый раз писать «переходник» между моделью и очередным сервисом дорого и с ростом их разнообразия затраты растут экспоненциально. Нужна стандартизация

[Model Context Protocol (MCP)](https://modelcontextprotocol.io) — это открытый протокол стандартизированного подключения LLM к инструментам и данным; его можно визуализировать как «USB порт для ИИ». 

Появляются серверы инструментов и источники данных, которые реализуют протокол один раз, после чего любой совместимый агент может ими пользоваться. Тема здесь не столько техническая, сколько системная: почему стандарт важен для масштаба — он развязывает интеграцию и приложение, и за счёт сетевого эффекта порождает целую экосистему переиспользуемых компонентов

Были попытки навязать войну стандартов: A2A от Google, но на текущий момент пока стнадартом ялвяется MCP

### Фреймворки
Инструмент нужно выбирать под задачу, а не привязываться к одному. Полезно держать в голове грубую карту:
- LangGraph - про управление и контроль через графы состояний; силён там, где важны циклы и явная логика переходов
- LlamaIndex ([github](https://github.com/run-llama/llama_index)) — про данные и RAG; силён в индексации и извлечении знаний.
- AutoGen - про мульти-агентное взаимодействие в формате разговора
- CrewAI ([github](https://github.com/crewAIInc/crewAI)) — про ролевые «команды» агентов с понятным разделением ролей

Иногда фреймворк не нужен вовсе. Anthropic в своем эссе [«Building Effective Agents»](https://www.anthropic.com/engineering/building-effective-agents) рекомендуют всегда начинать с простого - прямых вызовов модели - и добавлять сложность только тогда, когда она реально окупается

---



## Промышленные системы

Мир продакшена кардинально отличается от мира академии. Здесь на первый план выходят иные требования. К основным можно отнести: оценку качества, грамотный мониторинг, обеспечение безопасности и оптимизация процессов

### Оценка
Без измерения невозможна осмысленная итерация. Сложность в том, что агент недетерминирован, проходит много шагов и часто заслуживает «частичного зачёта», — простой accuracy здесь не работает

В арсенале 
— метрики успешности задач (довёл ли агент дело до конца) 
- LLM-as-a-judge (использовать другую модель как оценщика качества)
- специализированные бенчмарки:
    - [WebArena](https://arxiv.org/abs/2307.13854) (2023) для задач в вебе,
    - [GAIA](https://arxiv.org/abs/2311.12983) для ассистентов общего назначения,
    - [τ-bench (TauBench)](https://arxiv.org/abs/2406.12045) для взаимодействия с инструментами и пользователем в реалистичных доменах.

Отдельно стоит оценка RAG-компонентов — например, через [RAGAS](https://aclanthology.org/2024.eacl-demo.16), который измеряет качество извлечения и достоверность ответа

### Наблюдаемость и отладка
Нужно видеть, что происходит внутри многошагового процесса. Базовые инструменты — трейсинг всех вызовов (модели и инструментов), логирование шагов рассуждения и средства отладки недетерминированных сценариев, где один и тот же вход может приводить к разным траекториям

### Безопасность
Агент, способный выполнять реальные действия в среде — это реальная угроза и здесь ставки выше, чем у обычного чат-бота. Они могут быть преднамеренными — **prompt injection** и **jailbreak**: вредоносные инструкции, спрятанные во входных данных или на веб-странице, которые перехватывают поведение агента. Или непреднамеренными - забыли закрыть и агент при решении совей задачи воспользовался этим бэкдором. Для защиты использует те же принципы, как в кибербезопсаности: изоляция инструментов (агенты имеют доступ только к песочнице - изолированной копии продакшена), принцип минимальных прав (агент получает ровно те доступы, что нужны) и ограничение области действия агента (у агента набора интсрументов)

### Оптимизация

Чтобы агента можно было реально развернуть, он должен быть достаточно дешёвым и быстрым. Главные рычаги — контроль стоимости и латентности, кэширование (в том числе кэш промптов, чтобы не пересчитывать повторяющийся контекст), выбор модели под подзадачу (мелкая быстрая модель на простые шаги, крупная — только там, где нужна) и маршрутизация запросов между моделями разной мощности

### Обучение с подкреплением для агентов

Если в части 1 мы учили модель рассуждать с помощью RL, то теперь та же логика применяется к агентам целиком: вместо ручной настройки промптов и оркестрации — обучение агента через RL на взаимодействии со средой, end-to-end. Сюда же относятся самоулучшающиеся агенты (self improving agents) и множество открытых проблем (стабильность обучения, награды на длинном горизонте, перенос между задачами). 

Обзорная статья — [«The Landscape of Agentic Reinforcement Learning for LLMs: A Survey»](https://arxiv.org/abs/2509.02547) (2025)

---



## Резюме главы

Итого мы научились 
- заставлять модель думать (CoT) →
- думать надёжнее через поиск по вариантам (Self-Consistency, ToT) →
- обучать думать (reasoning-модели, RLVR) →
- заземлили мышление на действие (ReAct) →
- формализовали действия как вызовы инструментов (Toolformer, function calling) →
- внесли структуру через планирование, дали агенту **непрерывность** через память (MemGPT) и **способность улучшаться** через рефлексию (Reflexion, Voyager) →
- распределили работу между агентами (AutoGen, MetaGPT) →
- взяли поток под контроль через графовую оркестрацию (LangGraph, паттерны Anthropic) →
- стандартизировали подключение (MCP) →
- наконец, закалили систему для продакшена через оценку, наблюдаемость, безопасность и оптимизацию →
- агентов начинают обучать целиком через RL

Поле меняется быстро, поэтому статичный список устаревает. Полезны постоянно обновляемые источники: ежегодные обзоры статей Себастьяна Рашки, reading list от Latent.Space и GitHub-репозитории `zjunlp/LLMAgentPapers` и `AGI-Edgerunners/LLM-Agents-Papers`.